# 🔍 Tahap 3: Case Retrieval

**Tujuan:** Temukan kasus lama yang paling mirip dengan *query* kasus baru.

**Input:** `SUBCPMK-3/processed/cases.csv` (hasil Tahap 2)
**Output:**
- `03_retrieval.py` (script) / cell notebook tahap retrieval
- Fungsi `retrieve(query, k)` yang sudah teruji
- `SUBCPMK-3/eval/queries.json`  (≡ `/data/eval/queries.json` pada spesifikasi tugas)

---
| Langkah | Isi |
|---------|-----|
| **i**   | Representasi vektor — **TF-IDF** + **BERT Embedding** (IndoBERT `indobenchmark/indobert-base-p1`) |
| **ii**  | Splitting data — train/test **80:20** (stratified per `jenis_perkara`) |
| **iii** | Model retrieval — **SVM** & **Naive Bayes** (TF-IDF) + **IndoBERT** embedding (cosine) |
| **iv**  | Fungsi `retrieve(query, k)` — preprocess → vektor → cosine-similarity → top-k |
| **v**   | Pengujian awal — 8–10 query uji + ground-truth di `queries.json` |
| **vi**  | Output — script `03_retrieval.py`, fungsi teruji, file `queries.json` |


## Cell 17 — Install Library (Tahap 3)

In [ ]:
# transformers + torch untuk IndoBERT; scikit-learn untuk TF-IDF/SVM/NB
# ⚠️  Pin Pillow<12 — Pillow 12.x di Colab bikin 'from PIL._typing import _Ink'
#     GAGAL, sehingga AutoModel.from_pretrained (Cell 21) crash saat memuat torchvision/PIL.
!pip install -q "pillow<12" scikit-learn transformers torch

import PIL
print('✅ scikit-learn, transformers, torch siap! | Pillow =', PIL.__version__)
if int(PIL.__version__.split('.')[0]) >= 12:
    print('⚠️  Pillow masih 12.x di kernel ini.')
    print('    → Runtime > Restart session, lalu jalankan ULANG sel ini sebelum lanjut.')
else:
    print('ℹ️  Jika sel ini BARU menurunkan versi Pillow (mis. dari 12.x),')
    print('    tetap Restart session sekali lalu jalankan ulang dari Cell 17.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 36.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.
✅ scikit-learn, transformers, torch siap! | Pillow = 11.3.0
ℹ️  Jika sel ini BARU menurunkan versi Pillow (mis. dari 12.x),
    tetap Restart session sekali lalu jalankan ulang dari Cell 17.


## Cell 18 — Konfigurasi Path & Muat `cases.csv`

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, re, json

# ── Path (Colab + Google Drive) ─────────────────────────────────
PROCESSED_DIR = Path("/content/drive/MyDrive/SUBCPMK-3/processed")
CSV_IN        = PROCESSED_DIR / "cases.csv"

# eval/ ≡ /data/eval pada spesifikasi tugas
EVAL_DIR  = Path("/content/drive/MyDrive/SUBCPMK-3/eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)
QUERIES_JSON = EVAL_DIR / "queries.json"
SCRIPT_OUT   = Path("/content/drive/MyDrive/SUBCPMK-3/03_retrieval.py")

# ── Muat cases.csv ──────────────────────────────────────────────
df = pd.read_csv(CSV_IN, encoding="utf-8-sig", dtype=str).fillna("")
df = df.reset_index(drop=True)

print(f"✅ cases.csv dimuat — {len(df)} kasus, {len(df.columns)} kolom")
print(f"   Kolom: {list(df.columns)}")
print(f"📂 queries.json -> {QUERIES_JSON}")
print(f"📂 03_retrieval.py -> {SCRIPT_OUT}")

✅ cases.csv dimuat — 50 kasus, 20 kolom
   Kolom: ['case_id', 'source_file', 'nomor_perkara', 'tanggal_putusan', 'jenis_perkara', 'pengadilan', 'hakim_ketua', 'pihak1_label', 'pihak1', 'pihak2_label', 'pihak2', 'pasal', 'dakwaan_ringkas', 'tuntutan', 'barang_bukti', 'pertimbangan_hukum', 'amar_putusan', 'word_count', 'top_terms', 'qa_pairs']
📂 queries.json -> /content/drive/MyDrive/SUBCPMK-3/eval/queries.json
📂 03_retrieval.py -> /content/drive/MyDrive/SUBCPMK-3/03_retrieval.py


## Cell 18a — Re-Ekstraksi Amar & Dakwaan dari Teks Mentah (Tahap 2 diperbaiki)

> Ekstraksi awal Tahap 2 sering salah mengambil seksi PDF sehingga `amar_putusan` & `dakwaan_ringkas`
> tidak akurat. Cell ini **membaca ulang file `.txt` mentah** (output Tahap 1) dan mengekstrak seksi
> putusan dengan benar (setelah penanda **MENGADILI** untuk amar, narasi **"Bahwa terdakwa…"** untuk dakwaan),
> menangani huruf ber-spasi PDF dan terbilang ("lima tahun").
>
> **Jika folder teks mentah tidak ditemukan, cell ini dilewati otomatis** dan kolom lama tetap dipakai.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Fungsi ekstraksi seksi putusan (tertanam — tidak perlu file terpisah)
# ══════════════════════════════════════════════════════════════
import re

# ── Terbilang -> angka (untuk "lima tahun" tanpa digit) ─────────────────────
TERBILANG = {
    'nol': 0, 'satu': 1, 'dua': 2, 'tiga': 3, 'empat': 4, 'lima': 5,
    'enam': 6, 'tujuh': 7, 'delapan': 8, 'sembilan': 9, 'sepuluh': 10,
    'sebelas': 11, 'duabelas': 12, 'dua belas': 12, 'tiga belas': 13,
    'empat belas': 14, 'lima belas': 15, 'enam belas': 16, 'tujuh belas': 17,
    'delapan belas': 18, 'sembilan belas': 19, 'dua puluh': 20,
}


def despace(text):
    """Gabungkan huruf ber-spasi: 'M E N G A D I L I' -> 'MENGADILI'."""
    return re.sub(r'\b(?:[A-Za-z]\s){2,}[A-Za-z]\b',
                  lambda m: m.group(0).replace(' ', ''), text)


def normalize_text(text):
    """Bersihkan teks mentah putusan agar siap diparsing."""
    t = despace(str(text))
    # buang header/footer berulang & nomor halaman
    t = re.sub(r'(?i)direktori putusan\s+mahkamah\s+agung[^\n]*', ' ', t)
    t = re.sub(r'(?i)halaman\s+\d+\s+dari\s+\d+[^\n]*', ' ', t)
    t = re.sub(r'(?i)disclaimer[^\n]*', ' ', t)
    t = re.sub(r'(?i)email\s*:\s*\S+|telp\s*:\s*\S+', ' ', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()


def _section(text, start_pats, end_pats):
    """Ambil teks antara penanda awal (start_pats) dan penanda akhir (end_pats)."""
    low = text.lower()
    s = None
    for p in start_pats:
        m = re.search(p, low)
        if m:
            s = m.end()
            break
    if s is None:
        return ""
    e = len(text)
    for p in end_pats:
        m = re.search(p, low[s:])
        if m:
            e = s + m.start()
            break
    return text[s:e].strip()


# ── Ekstraksi AMAR PUTUSAN ──────────────────────────────────────────────────
def extract_amar(text):
    """Ambil seksi setelah 'MENGADILI' (amar putusan)."""
    t = normalize_text(text)
    amar = _section(
        t,
        start_pats=[r'\bmengadili\b\s*:?', r'\bm\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i\b'],
        end_pats=[r'demikian(?:lah)?\s+di(?:putus|adili)', r'\bditetapkan\s+di\b',
                  r'panitera\s+pengganti', r'hakim\s+ketua\b.*?hakim'],
    )
    # Bila MENGADILI tak ketemu, cari langsung kalimat vonis
    if len(amar) < 15:
        m = re.search(r'(menjatuhkan\s+pidana[^.]{0,200})', t, re.I)
        amar = m.group(1) if m else ""
    return amar.strip()


# ── Ekstraksi DAKWAAN ───────────────────────────────────────────────────────
def extract_dakwaan(text):
    """Ambil ringkasan dakwaan (narasi 'Bahwa terdakwa ...')."""
    t = normalize_text(text)
    dak = _section(
        t,
        start_pats=[r'\bbahwa\s+(?:ia\s+)?terdakwa\b', r'\bdakwaan\b\s*:?',
                    r'\b(?:primair|kesatu|pertama)\b\s*:?'],
        end_pats=[r'\btuntutan\b', r'\bmenuntut\b', r'pembacaan\s+tuntutan',
                  r'\bmenimbang\b', r'fakta\s+(?:hukum|persidangan)',
                  r'\bmengadili\b', r'\bmemutuskan\b'],
    )
    # ambil 1-2 kalimat pertama yang faktual
    sents = re.split(r'(?<=[.;])\s+', dak)
    keep = [s for s in sents if len(s) > 25][:3]
    return ' '.join(keep).strip()[:500]


# ── Parsing outcome dari amar ───────────────────────────────────────────────
def _terbilang_to_int(word):
    word = word.strip().lower()
    return TERBILANG.get(word)


def parse_penjara_bulan(amar):
    """Lama pidana penjara dalam BULAN. Tangani digit & terbilang."""
    t = str(amar).lower()
    if 'seumur hidup' in t or 'pidana mati' in t:
        return 999  # penanda hukuman maksimal
    # Pola digit: "5 (lima) tahun [dan 6 (enam) bulan]"
    m = re.search(r'penjara\s+(?:selama\s+)?(\d+)\s*(?:\([^)]*\)\s*)?tahun'
                  r'(?:\s*(?:dan\s+)?(\d+)\s*(?:\([^)]*\)\s*)?bulan)?', t)
    if m:
        return int(m.group(1)) * 12 + (int(m.group(2)) if m.group(2) else 0)
    # Pola terbilang: "penjara selama lima tahun"
    m = re.search(r'penjara\s+(?:selama\s+)?([a-z]+(?:\s+belas|\s+puluh)?)\s+tahun', t)
    if m:
        n = _terbilang_to_int(m.group(1))
        if n is not None:
            mb = re.search(r'(\w+(?:\s+belas)?)\s+bulan', t)
            bulan = _terbilang_to_int(mb.group(1)) if mb else 0
            return n * 12 + (bulan or 0)
    # Hanya bulan
    m = re.search(r'penjara\s+(?:selama\s+)?(\d+)\s*(?:\([^)]*\)\s*)?bulan', t)
    if m:
        return int(m.group(1))
    return None


def parse_denda(amar):
    m = re.search(r'denda\s+(?:sebesar\s+|sejumlah\s+)?rp[\s.]*([\d.\s]{3,})', str(amar).lower())
    if not m:
        return ''
    val = re.sub(r'\s+', '', m.group(1)).strip('.')
    return f'Rp {val}' if val else ''


def parse_verdict(amar):
    t = str(amar).lower()
    if not t.strip():
        return None
    if re.search(r'\bbebas\b|tidak\s+terbukti\s+secara\s+sah', t):
        return 'Bebas'
    if re.search(r'lepas\s+dari\s+segala\s+tuntutan', t):
        return 'Lepas'
    if re.search(r'rehabilitasi', t):
        return 'Rehabilitasi'
    if re.search(r'pidana\s+penjara|penjara\s+selama|seumur\s+hidup', t) \
       or parse_penjara_bulan(t) is not None:
        return 'Pidana penjara'
    return 'Lainnya'


def extract_all(raw_text):
    """Ekstrak amar + dakwaan + outcome dari satu teks putusan mentah."""
    amar = extract_amar(raw_text)
    dakwaan = extract_dakwaan(raw_text)
    return {
        'amar_putusan': amar,
        'dakwaan_ringkas': dakwaan,
        'verdict': parse_verdict(amar),
        'penjara_bulan': parse_penjara_bulan(amar),
        'denda': parse_denda(amar),
        'valid_amar': bool(parse_verdict(amar)),
    }


# ── Re-ekstraksi dari teks mentah Tahap 1 ───────────────────────
from pathlib import Path

# Sesuaikan dengan lokasi output Tahap 1 Anda:
RAW_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/SUBCPMK-3/raw"),
    Path("/content/drive/MyDrive/SUBCPMK-3/data/raw"),
    Path("/content/drive/MyDrive/SUBCPMK-3/NARKOTIKA PN DENPASAR/raw"),
]
RAW_DIR = next((p for p in RAW_DIR_CANDIDATES if p.exists()), None)

if RAW_DIR is None:
    print("⚠️  Folder teks mentah (.txt) tidak ditemukan -> re-ekstraksi DILEWATI.")
    print("    Kolom amar/dakwaan lama tetap dipakai. Set RAW_DIR manual bila perlu.")
else:
    files = sorted(RAW_DIR.glob("case_*.txt"))
    print(f"📂 RAW_DIR: {RAW_DIR} ({len(files)} file)")
    n_valid = 0
    for f in files:
        cid = f.stem
        raw = f.read_text(encoding="utf-8", errors="ignore")
        ex  = extract_all(raw)
        mask = df["case_id"] == cid
        if mask.any():
            if ex["amar_putusan"]:
                df.loc[mask, "amar_putusan"] = ex["amar_putusan"]
            if ex["dakwaan_ringkas"]:
                df.loc[mask, "dakwaan_ringkas"] = ex["dakwaan_ringkas"]
            if ex["valid_amar"]:
                n_valid += 1
    print(f"✅ Re-ekstraksi selesai — {n_valid}/{len(files)} kasus punya amar valid")
    # tampilkan contoh
    for f in files[:3]:
        ex = extract_all(f.read_text(encoding="utf-8", errors="ignore"))
        print(f"   [{f.stem}] {ex['verdict']} | {ex['penjara_bulan']} bln | {ex['amar_putusan'][:60]}")

📂 RAW_DIR: /content/drive/MyDrive/SUBCPMK-3/raw (50 file)
✅ Re-ekstraksi selesai — 50/50 kasus punya amar valid
   [case_001] Pidana penjara | 87 bln | perkara pidadna dengan acara pemeriksaan biasa dalam tingkat
   [case_002] Rehabilitasi | 36 bln | perkara pidana dengan agcara pemeriksaan biasa dalam tingkat
   [case_003] Pidana penjara | 54 bln | perkara pidana dengan aacara pemeriksaan biasa dalam tingkat


## Cell 18b — Pembersihan Data (`dakwaan_ringkas`, `barang_bukti`, `amar_putusan`, `top_terms`)

> Kolom-kolom di `cases.csv` sering mengandung artefak dari ekstraksi PDF (huruf tunggal, header halaman,
> potongan tidak bermakna). Cell ini membersihkan dan merekonstruksi kolom-kolom kunci **sebelum** dipakai
> untuk retrieval & prediksi. Jalankan sekali setelah memuat `cases.csv`.

In [ ]:
import re

# ── Noise words: angka tulisan + stopword umum ──────────────────
NOISE_NUMS  = {'satu','dua','tiga','empat','lima','enam','tujuh','delapan',
               'sembilan','sepuluh','sebelas','puluh','ratus','ribu'}
NOISE_STOP  = {'yang','dan','di','ke','dari','dengan','untuk','dalam','adalah',
               'pada','oleh','ini','itu','tidak','atau','juga','akan','telah',
               'sebagai','bahwa','tersebut','kepada','para','serta','dapat',
               'ada','sudah','atas','bagi','agar','warna','tanggal','nomor',
               'namun','buah','jenis','berisi','sediaan','halaman','perkara',
               'barang','bukti','berikut','saat'}
NOISE_WORDS = NOISE_NUMS | NOISE_STOP

def clean_noise(text):
    """Bersihkan artefak PDF: huruf tunggal, nomor halaman, header MA."""
    t = str(text).lower()
    t = re.sub(r'(?<=[^a-z])[a-z](?=[^a-z])', ' ', t)     # huruf tunggal
    t = re.sub(r'halaman\s+\d+\s*(dari\s*\d+)?', ' ', t)   # nomor halaman
    t = re.sub(r'direktori putusan\s+mahkamah\s+agung[^.]*', ' ', t)
    t = re.sub(r'putusan nomor\s+\S+', ' ', t)
    t = re.sub(r'\d+/pid\.sus/\d+/pn\s*\w*', ' ', t)       # kode perkara
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()

# ── 1. Filter top_terms ────────────────────────────────────────
def filter_terms(terms_str):
    terms = [t.strip() for t in str(terms_str).split(',')]
    return ', '.join(t for t in terms if t and t.lower() not in NOISE_WORDS and len(t) > 2)[:300]

# ── 2. Bersihkan barang_bukti ──────────────────────────────────
def clean_bb(text):
    t = clean_noise(text)
    segs = re.split(r'[;.\n]', t)
    keep = [s.strip() for s in segs if len(s.strip()) > 20 and
            re.search(r'narkotika|shabu|sabu|ganja|ekstasi|pil|gram|butir|'
                      r'plastik|klip|bungkus|paket|terdakwa|'
                      r'diamankan|ditemukan|disita|membawa|menyimpan', s, re.I)]
    return '. '.join(keep[:3]).strip() or t[:300]

# ── 3. Rebuild dakwaan_ringkas dari pihak1 + top_terms + barang_bukti
def rebuild_dakwaan(row):
    nama = str(row.get('pihak1','')).strip()
    terms = filter_terms(row.get('top_terms',''))
    bb = clean_bb(row.get('barang_bukti',''))
    parts = []
    if nama and len(nama) > 3 and not re.search(r'sejak|tanggal|\d{4}', nama):
        parts.append(f"terdakwa {nama.lower()}")
    if terms:
        parts.append(f"terkait {terms}")
    if bb and len(bb) > 20:
        parts.append(bb)
    return '. '.join(parts)[:450] if parts else ''

# ── 4. Infer amar_putusan dari teks yang ada + pasal ──────────
def fix_amar(row):
    # FIX: JANGAN ambil dari 'tuntutan' (permintaan JPU, bukan putusan hakim)
    #      maupun 'barang_bukti' agar amar tidak terkontaminasi.
    all_text = ' '.join(str(row.get(c,'')) for c in
               ['amar_putusan','pertimbangan_hukum'])
    # Cari kalimat verdict nyata di kolom putusan saja
    m = re.search(r'(menjatuhkan\s+pidana\s+penjara\s+(?:selama\s+)?\d+[^.;]{0,80})', all_text, re.I)
    if m: return m.group(1).strip()
    m = re.search(r'(pidana\s+penjara\s+selama\s+\d+[^.;]{0,80})', all_text, re.I)
    if m: return m.group(1).strip()
    m = re.search(r'(pidana\s+penjara\s+seumur\s+hidup[^.;]{0,40})', all_text, re.I)
    if m: return m.group(1).strip()
    m = re.search(r'(menjalani\s+rehabilitasi[^.;]{0,100})', all_text, re.I)
    if m: return 'menetapkan terdakwa ' + m.group(1).strip()
    m = re.search(r'(rehabilitasi\s+\w+[^.;]{0,80})', all_text, re.I)
    if m: return 'menetapkan terdakwa menjalani ' + m.group(1).strip()
    # ★ TIDAK fabrikasi: bila amar nyata tak ditemukan -> kosong
    return ''

print("🧹 Membersihkan kolom data ...")
df['top_terms']       = df['top_terms'].apply(filter_terms)
df['barang_bukti']    = df['barang_bukti'].apply(clean_bb)
df['dakwaan_ringkas'] = df.apply(rebuild_dakwaan, axis=1)
df['amar_putusan']    = df.apply(fix_amar, axis=1)

# Laporan
for col in ['dakwaan_ringkas','barang_bukti','amar_putusan','top_terms']:
    ok = (df[col].str.len() > 30).sum()
    print(f"  {col:<22} meaningful: {ok}/{len(df)}")
print("\nContoh dakwaan_ringkas[0]:", df.iloc[0]['dakwaan_ringkas'][:150])
print("Contoh amar_putusan[0] :", df.iloc[0]['amar_putusan'][:120])
# ★ Simpan data bersih ke disk agar cell setelahnya memuat versi yang sama
CSV_CLEANED = PROCESSED_DIR / "cases_cleaned.csv"
df.to_csv(CSV_CLEANED, index=False, encoding="utf-8-sig")
print(f"💾 Data bersih disimpan -> {CSV_CLEANED}")
# ── Helper kategori (SUMBER TUNGGAL, dipakai Tahap 3 & 5) ──
def pasal_label(s):
    m = re.search(r'pasal\s+(\d+)(?:\s+ayat\s*\(?(\d+)\)?)?', str(s).lower())
    if not m: return "Lainnya"
    return f"Pasal {m.group(1)}" + (f" ayat {m.group(2)}" if m.group(2) else "")

def build_case_cat(dframe):
    lj = dframe['jenis_perkara'].replace('', 'Lainnya')
    if lj.nunique() >= 2:
        return dict(zip(dframe['case_id'], lj)), 'jenis_perkara'
    return dict(zip(dframe['case_id'], dframe['pasal'].map(pasal_label))), 'pasal_utama'

print("✅ Pembersihan selesai")

🧹 Membersihkan kolom data ...
  dakwaan_ringkas        meaningful: 50/50
  barang_bukti           meaningful: 39/50
  amar_putusan           meaningful: 43/50
  top_terms              meaningful: 48/50

Contoh dakwaan_ringkas[0]: terdakwa aldo almi rhody. terkait narkotika, plastik, muding, denpasar. narkotika di dalam tas minibelt warna coklat yang dibawa oleh terdakwa berupa 
Contoh amar_putusan[0] : pidana penjara selama 7 (tujuh) tahun 3 (tiga) bulan dikurangi selama terdakwa berada dalam tahanan sem
💾 Data bersih disimpan -> /content/drive/MyDrive/SUBCPMK-3/processed/cases_cleaned.csv
✅ Pembersihan selesai


## Cell 19 — Bangun Representasi Teks per Kasus (corpus)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Gabungkan kolom-kolom penting menjadi SATU representasi teks per
# kasus. Inilah "dokumen" yang akan divektorkan & dicari saat retrieval.
# ══════════════════════════════════════════════════════════════

TEXT_COLS = [
    'jenis_perkara', 'pasal', 'pengadilan',
    'dakwaan_ringkas', 'tuntutan', 'barang_bukti',
    'pertimbangan_hukum', 'amar_putusan', 'top_terms',
]
TEXT_COLS = [c for c in TEXT_COLS if c in df.columns]

def build_repr(row):
    parts = [str(row.get(c, '')) for c in TEXT_COLS]
    return ' '.join(p for p in parts if p).strip()

def preprocess(text: str) -> str:
    """Pra-pemrosesan ringan: lowercase + rapikan spasi/simbol.
    Konsisten dengan output Tahap 1 yang sudah lowercase."""
    t = str(text).lower()
    t = re.sub(r'[^a-z0-9\s./()-]', ' ', t)   # buang simbol aneh
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

df['text_repr'] = df.apply(build_repr, axis=1).map(preprocess)
df['label']     = df['jenis_perkara'].replace('', 'Lainnya')   # label klasifikasi

CASE_IDS = df['case_id'].tolist()
CORPUS   = df['text_repr'].tolist()

empty = sum(1 for c in CORPUS if not c)
avg   = int(np.mean([len(c.split()) for c in CORPUS])) if CORPUS else 0
print(f"✅ {len(CORPUS)} representasi teks dibangun")
print(f"   Kosong: {empty} | Rata-rata panjang: {avg} kata")
print(f"\n   Contoh corpus[0] ({CASE_IDS[0]}):")
print(f"   {CORPUS[0][:240]}...")

✅ 50 representasi teks dibangun
   Kosong: 0 | Rata-rata panjang: 177 kata

   Contoh corpus[0] (case_001):
   pidana khusus pasal 114 pasal 112 pasal 111 ayat (1) pasal 182 ayat (4) pasal 114 ayat (2) pasal 112 ayat (2) pasal 7 pasal 36 pasal 38 pasal 41 pengadilan negeri denpasar terdakwa aldo almi rhody. terkait narkotika plastik muding denpasar....


## Cell 20 — i.1 Representasi Vektor: **TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np, re

STOPWORDS_ID = [
    'yang','dan','di','ke','dari','dengan','untuk','dalam','adalah',
    'pada','oleh','ini','itu','tidak','atau','juga','akan','telah',
    'sebagai','bahwa','tersebut','kepada','para','serta','dapat','ada',
    'sudah','atas','bagi','agar','lebih','saat','hal','tentang','tahun',
    'maka','karena','hingga','antara','terhadap','setelah','sebelum',
]

# ══════════════════════════════════════════════════════════════
# ★ FIX LEAKAGE: SPLIT DULU, lalu fit TF-IDF HANYA pada train.
# Stratify pada label klasifikasi (pasal) agar train memuat semua kelas.
# ══════════════════════════════════════════════════════════════
def _pasal_coarse(s):
    m = re.search(r'pasal\s+(\d+)', str(s).lower())
    return f"Pasal {m.group(1)}" if m else "Lainnya"

_lj = df['jenis_perkara'].replace('', 'Lainnya')
y_strat = (_lj.values if _lj.nunique() >= 2 else df['pasal'].map(_pasal_coarse).values)

TEST_SIZE, RANDOM_STATE = 0.20, 42
idx     = np.arange(len(df))
strat   = y_strat if min(Counter(y_strat).values()) >= 2 else None
if strat is None:
    print("⚠️  Sebagian kelas <2 anggota -> split acak tanpa stratify")
idx_train, idx_test = train_test_split(idx, test_size=TEST_SIZE,
                                       random_state=RANDOM_STATE, stratify=strat)

tfidf = TfidfVectorizer(lowercase=True, stop_words=STOPWORDS_ID,
                        ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)

# ★ Fit HANYA pada teks train (vocabulary tidak menyerap test)
tfidf.fit([CORPUS[i] for i in idx_train])

# Transform: SEMUA kasus (case base untuk retrieve) + train/test terpisah
X_tfidf = tfidf.transform(CORPUS)          # index case base (vocab dari train)
X_train = X_tfidf[idx_train]
X_test  = X_tfidf[idx_test]

sparsity = 100 * (1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]))
print(f"✅ TF-IDF di-fit pada {len(idx_train)} kasus TRAIN saja (tanpa leakage)")
print(f"   Index case base : {X_tfidf.shape[0]} kasus × {X_tfidf.shape[1]} fitur")
print(f"   Train={X_train.shape[0]} | Test={X_test.shape[0]} | sparsity={sparsity:.1f}%")
print("   ℹ️ Untuk deploy nyata, fit ulang TF-IDF pada SELURUH case base.")

⚠️  Sebagian kelas <2 anggota -> split acak tanpa stratify
✅ TF-IDF di-fit pada 40 kasus TRAIN saja (tanpa leakage)
   Index case base : 50 kasus × 695 fitur
   Train=40 | Test=10 | sparsity=86.1%
   ℹ️ Untuk deploy nyata, fit ulang TF-IDF pada SELURUH case base.


## Cell 21 — i.2 Representasi Vektor: **BERT Embedding (IndoBERT)**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

BERT_MODEL = "indobenchmark/indobert-base-p1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️  Memuat {BERT_MODEL} di {device}")

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert      = AutoModel.from_pretrained(BERT_MODEL).to(device).eval()

@torch.no_grad()
def embed_texts(texts, batch_size=8, max_len=256):
    """Encode list teks -> matriks embedding (mean-pooling token, L2-normalized)."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = [t if t else "kosong" for t in texts[i:i+batch_size]]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt").to(device)
        out  = bert(**enc).last_hidden_state          # (B, T, H)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)   # mean pooling
        mean = torch.nn.functional.normalize(mean, p=2, dim=1)     # L2-norm
        vecs.append(mean.cpu().numpy())
    return np.vstack(vecs)

X_bert = embed_texts(CORPUS)
print(f"✅ IndoBERT embedding: {X_bert.shape[0]} kasus × {X_bert.shape[1]} dimensi")

⚙️  Memuat indobenchmark/indobert-base-p1 di cpu ... (unduh sekali, agak lama)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


2026-06-16 10:55:44,349 | INFO | HTTP Request: HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-06-16 10:55:44,363 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-16 10:55:44,390 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/indobenchmark/indobert-base-p1/c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2/config.json "HTTP/1.1 200 OK"
2026-06-16 10:55:44,420 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/indobenchmark/indobert-base-p1/c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

2026-06-16 10:56:04,253 | INFO | HTTP Request: HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-06-16 10:56:04,320 | INFO | HTTP Request: GET https://huggingface.co/api/models/indobenchmark/indobert-base-p1 "HTTP/1.1 200 OK"
2026-06-16 10:56:04,458 | INFO | HTTP Request: GET https://huggingface.co/api/models/indobenchmark/indobert-base-p1/commits/main "HTTP/1.1 200 OK"
2026-06-16 10:56:04,523 | INFO | HTTP Request: GET https://huggingface.co/api/models/indobenchmark/indobert-base-p1/discussions?p=0 "HTTP/1.1 200 OK"
2026-06-16 10:56:04,621 | INFO | HTTP Request: GET https://huggingface.co/api/models/indobenchmark/indobert-base-p1/commits/refs%2Fpr%2F5 "HTTP/1.1 200 OK"
2026-06-16 10:56:04,693 | INFO | HTTP Request: HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/refs%2Fpr%2F5/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-06-16 10:56:05,347 | INFO | HTTP Request: HEAD https://huggingfa

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ IndoBERT embedding: 50 kasus × 768 dimensi


## Cell 22 — ii. Splitting Data (Train/Test **80:20**)

In [ ]:
from collections import Counter
# ══════════════════════════════════════════════════════════════
# ★ FIX: split sudah dibuat di Cell 20 (sebelum fit TF-IDF) agar
# konsisten & bebas leakage. Sel ini hanya menampilkan ringkasannya.
# ══════════════════════════════════════════════════════════════
labels = df['label'].values
print(f"✅ Split (dibuat di Cell 20, rasio {int((1-TEST_SIZE)*100)}:{int(TEST_SIZE*100)})")
print(f"   Train: {len(idx_train)} kasus | Test: {len(idx_test)} kasus")
print(f"   Distribusi label train: {dict(Counter(labels[idx_train]))}")
print(f"   Distribusi label test : {dict(Counter(labels[idx_test]))}")

✅ Split (dibuat di Cell 20, rasio 80:20)
   Train: 40 kasus | Test: 10 kasus
   Distribusi label train: {'Pidana Khusus': 40}
   Distribusi label test : {'Pidana Khusus': 10}


## Cell 23 — iii.1 Model Retrieval/Klasifikasi: **SVM & Naive Bayes** (TF-IDF)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB  # ★ lebih tepat utk TF-IDF berbobot
from sklearn.metrics import classification_report, accuracy_score
import re, numpy as np

# ══════════════════════════════════════════════════════════════
# Label klasifikasi (jenis_perkara bila >=2 kelas, else pasal)
# ★ FIX: pakai split idx_train/idx_test DARI Cell 20 (tidak split ulang)
# ══════════════════════════════════════════════════════════════
def pasal_label(s):
    m = re.search(r'pasal\s+(\d+)', str(s).lower())
    return f"Pasal {m.group(1)}" if m else "Lainnya"

label_jenis = df['jenis_perkara'].replace('', 'Lainnya')
label_pasal = df['pasal'].map(pasal_label)
if label_jenis.nunique() >= 2:
    y_all, LABEL_NAME = label_jenis.values, "jenis_perkara"
elif label_pasal.nunique() >= 2:
    y_all, LABEL_NAME = label_pasal.values, "pasal_utama"
    print("ℹ️  jenis_perkara 1 kelas -> label klasifikasi = PASAL utama")
else:
    y_all, LABEL_NAME = None, None
    print("⚠️  Dataset 1 kelas -> SVM/NB dilewati.")

clf_results = {}
if y_all is not None:
    # ★ Gunakan SPLIT YANG SAMA dari Cell 20
    Xtr, Xte = X_tfidf[idx_train], X_tfidf[idx_test]
    ytr, yte = y_all[idx_train],  y_all[idx_test]

    models = {"SVM (LinearSVC)": LinearSVC(C=1.0),
              "Naive Bayes (ComplementNB)": ComplementNB(alpha=0.3)}
    for name, clf in models.items():
        if len(set(ytr)) < 2:
            print(f"⚠️  {name} dilewati — train hanya 1 kelas."); continue
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        acc  = accuracy_score(yte, pred)
        clf_results[name] = (clf, acc)
        print("=" * 60)
        print(f"  {name} | label={LABEL_NAME} | akurasi TEST = {acc:.3f}")
        print("=" * 60)
        print(classification_report(yte, pred, zero_division=0))
    if clf_results:
        best_name = max(clf_results, key=lambda k: clf_results[k][1])
        best_clf  = clf_results[best_name][0]
        print(f"🏆 Terbaik: {best_name} ({clf_results[best_name][1]:.3f})")
print("ℹ️  Test set kecil (~7 kasus) -> angka indikatif; lihat K-Fold CV di Tahap 5.")

ℹ️  jenis_perkara 1 kelas -> label klasifikasi = PASAL utama
  SVM (LinearSVC) | label=pasal_utama | akurasi TEST = 0.700
              precision    recall  f1-score   support

   Pasal 112       1.00      0.50      0.67         2
   Pasal 113       0.00      0.00      0.00         0
   Pasal 114       1.00      0.83      0.91         6
   Pasal 127       1.00      1.00      1.00         1
    Pasal 25       0.00      0.00      0.00         1
   Pasal 609       0.00      0.00      0.00         0

    accuracy                           0.70        10
   macro avg       0.50      0.39      0.43        10
weighted avg       0.90      0.70      0.78        10

  Naive Bayes (ComplementNB) | label=pasal_utama | akurasi TEST = 0.400
              precision    recall  f1-score   support

   Pasal 111       0.00      0.00      0.00         0
   Pasal 112       1.00      0.50      0.67         2
   Pasal 113       0.00      0.00      0.00         0
   Pasal 114       1.00      0.33      0.50   

## Cell 24 — iii.2 Model Retrieval: **IndoBERT Embedding** (cosine)

Retrieval embedding memakai *cosine-similarity* langsung pada `X_bert` (seluruh case base).
Sebagai proxy kualitas, kita ukur **konsistensi label tetangga terdekat**: untuk tiap kasus,
apakah mayoritas *k*-tetangga terdekatnya berlabel sama. Makin tinggi = kasus sejenis makin berdekatan di ruang vektor.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def neighbor_label_consistency(X, labels, k=3):
    S = cosine_similarity(X)
    np.fill_diagonal(S, -1)                 # buang diri sendiri
    hit = 0
    for i in range(len(labels)):
        topk  = np.argsort(-S[i])[:k]
        votes = Counter(labels[topk])
        if votes.most_common(1)[0][0] == labels[i]:
            hit += 1
    return hit / len(labels)

# FIX: df["label"] sering hanya 1 kelas (semua "Pidana Khusus") sehingga
#      konsistensi tetangga selalu 1.0 (palsu). Gunakan kategori sebenarnya
#      (pasal utama bila jenis_perkara seragam) lewat build_case_cat().
_cat_map, _cat_name = build_case_cat(df)
labels_eval = np.array([_cat_map[cid] for cid in df["case_id"]])
print(f"  Perbandingan ruang vektor (konsistensi label tetangga, kategori={_cat_name}):")
for name, X in [("TF-IDF", X_tfidf), ("IndoBERT", X_bert)]:
    c1 = neighbor_label_consistency(X, labels_eval, k=1)
    c3 = neighbor_label_consistency(X, labels_eval, k=3)
    print(f"    {name:<10} @1={c1:.2f}   @3={c3:.2f}")

  Perbandingan ruang vektor (konsistensi label tetangga, kategori=pasal_utama):
    TF-IDF     @1=0.46   @3=0.48
    IndoBERT   @1=0.36   @3=0.32


## Cell 25 — iv. Fungsi `retrieve(query, k)`

In [ ]:
from typing import List, Dict
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query: str, k: int = 5, method: str = "tfidf") -> List[Dict]:
    """
    Temukan top-k kasus paling mirip dengan query.

    Parameter
    ---------
    query  : teks kasus baru (ringkasan fakta / pertanyaan)
    k      : jumlah kasus yang dikembalikan
    method : 'tfidf' (ruang fitur SVM/NB) atau 'bert' (IndoBERT embedding)

    Return : list dict [{rank, case_id, score, jenis_perkara, nomor_perkara}, ...]
    """
    # 1) Pra-pemrosesan query
    q = preprocess(query)

    # 2) Hitung vektor query  +  3) cosine-similarity ke semua case vectors
    if method == "tfidf":
        sims = cosine_similarity(tfidf.transform([q]), X_tfidf).ravel()
    elif method == "bert":
        sims = cosine_similarity(embed_texts([q]), X_bert).ravel()
    else:
        raise ValueError("method harus 'tfidf' atau 'bert'")

    # 4) Kembalikan top-k case_id
    top_idx = np.argsort(-sims)[:k]
    return [
        {
            "rank"          : rank,
            "case_id"       : df.iloc[i]["case_id"],
            "score"         : round(float(sims[i]), 4),
            "jenis_perkara" : df.iloc[i]["jenis_perkara"],
            "nomor_perkara" : df.iloc[i]["nomor_perkara"],
        }
        for rank, i in enumerate(top_idx, start=1)
    ]

# ── Uji cepat ───────────────────────────────────────────────────
demo_q = "terdakwa kepemilikan narkotika golongan i jenis sabu melanggar pasal 112"
print("Query:", demo_q, "\n")
for m in ("tfidf", "bert"):
    print(f"── method = {m} ──")
    for r in retrieve(demo_q, k=3, method=m):
        print(f"  #{r['rank']} {r['case_id']} (score={r['score']}) "
              f"| {r['jenis_perkara']} | {r['nomor_perkara']}")
    print()

Query: terdakwa kepemilikan narkotika golongan i jenis sabu melanggar pasal 112 

── method = tfidf ──
  #1 case_040 (score=0.2744) | Pidana Khusus | 731/pid.sus/2023/pn dps
  #2 case_006 (score=0.2547) | Pidana Khusus | 1066/pid.sus/2023/pn dps
  #3 case_041 (score=0.2352) | Pidana Khusus | 

── method = bert ──
  #1 case_017 (score=0.61) | Pidana Khusus | 249/pid.sus/2026/pn dps
  #2 case_040 (score=0.6094) | Pidana Khusus | 731/pid.sus/2023/pn dps
  #3 case_009 (score=0.6018) | Pidana Khusus | 1078/pid.sus/2024/pn dps



## Cell 26 — v. Pengujian Awal: Bangun `queries.json` (NO-LEAK)

Query dibuat **otomatis** dari `dakwaan_ringkas` + `barang_bukti` (kolom faktual saja),
dengan **nomor pasal dihapus** via regex. `ground_truth_case_id` terisi otomatis = `case_id` kasus sumbernya.
Ini menghindari *data leakage* yang membuat skor TF-IDF artifisial tinggi (1.0).

In [ ]:
import json, random, re
random.seed(42)

# ══════════════════════════════════════════════════════════════
# Query NO-LEAK v4: sumber = dakwaan_ringkas + barang_bukti SAJA.
# TIDAK pakai text_repr/top_terms (top_terms diekstrak dari dok itu sendiri
# -> bocor & mendistorsi skor). Hapus nama terdakwa + pasal + golongan.
# ══════════════════════════════════════════════════════════════

def make_query_noleak(row):
    # ★ Sumber faktual saja — BUKAN text_repr (yg memuat top_terms doc itu sendiri)
    src = " ".join([str(row.get('dakwaan_ringkas', '')),
                    str(row.get('barang_bukti', ''))])
    # 1) Hapus nama terdakwa (identifier unik)
    nama = str(row.get('pihak1', '')).strip()
    if nama and len(nama) > 3:
        for part in nama.lower().split():
            if len(part) > 3:
                src = src.replace(part, '')
    # 2) Hapus pola "pasal NNN (ayat X)" & golongan (label, bukan fakta)
    src = re.sub(r'pasal\s+\d+(\s*ayat\s*[\d()]+)?', '', src, flags=re.IGNORECASE)
    src = re.sub(r'golongan\s+[iI]{1,3}', '', src, flags=re.IGNORECASE)
    return preprocess(src)

cand = list(range(len(df)))
random.shuffle(cand)
chosen = cand[:10]

queries = []
for qid, i in enumerate(chosen, start=1):
    row  = df.iloc[i]
    qtxt = make_query_noleak(row)
    if len(qtxt.strip()) < 20:
        qtxt = preprocess(str(row.get('barang_bukti','')))[:200]
    queries.append({
        "query_id"             : f"q{qid:02d}",
        "query"                : qtxt,
        "ground_truth_case_id" : row['case_id'],
        "catatan"              : "noleak-v4: dakwaan+barang_bukti, tanpa text_repr/top_terms",
    })

payload = {
    "deskripsi" : "Query uji no-leak v4",
    "dibuat"    : pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "jumlah"    : len(queries),
    "queries"   : queries,
}
QUERIES_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ {len(queries)} query no-leak v4 disimpan -> {QUERIES_JSON}")
for q in queries[:3]:
    print(f"  {q['query_id']} | GT={q['ground_truth_case_id']:<10} | {q['query'][:80]}...")
print("  ...")

✅ 10 query no-leak v4 disimpan -> /content/drive/MyDrive/SUBCPMK-3/eval/queries.json
  q01 | GT=case_026   | terdakwa . terkait narkotika gram berat shabu plastik bersih kode. tersebut dite...
  q02 | GT=case_024   | terdakwa afi . terkait narkotika berat gram plastik klip denpasar sabu. 993/2024...
  q03 | GT=case_020   | terdakwa i . terkait gram kode netto brutto paket narkotika shabu. narkotika jen...
  ...


## Cell 27 — Evaluasi `retrieve()` pada `queries.json` (Hit@k, Recall@k, MRR)

In [ ]:
# Kategori dari helper bersama (Cell 18b) — tanpa duplikasi
CASE_CAT, CAT_NAME = build_case_cat(df)

def evaluate(queries, k=5, method="tfidf"):
    """★ LEAVE-ONE-OUT + match KATEGORI (konsisten dgn evaluasi Tahap 5)."""
    hit, rr = 0, 0.0
    rows = []
    for q in queries:
        gt  = q["ground_truth_case_id"]
        raw = retrieve(q["query"], k=k+1, method=method)
        ids = [r["case_id"] for r in raw if r["case_id"] != gt][:k]   # exclude diri sendiri
        gt_cat = CASE_CAT.get(gt, "NONE")
        cats   = [CASE_CAT.get(c, "NONE") for c in ids]
        rank   = next((j+1 for j, c in enumerate(cats) if c == gt_cat), None)
        hit += int(rank is not None)
        rr  += (1.0/rank) if rank else 0.0
        rows.append((q["query_id"], gt, gt_cat, rank if rank else "—", ids[0] if ids else "-"))
    n = max(len(queries), 1)
    return {f"Hit@{k}": round(hit/n, 3), "MRR": round(rr/n, 3), "detail": rows}

qs = json.loads(QUERIES_JSON.read_text(encoding="utf-8"))["queries"]
K  = 5
print("ℹ️  Evaluasi LEAVE-ONE-OUT berbasis KATEGORI (konsisten dgn Tahap 5)\n")
for m in ("tfidf", "bert"):
    r = evaluate(qs, k=K, method=m)
    print("=" * 70)
    print(f"  METHOD={m.upper():<7} | Hit@{K}(kategori)={r[f'Hit@{K}']} | MRR={r['MRR']}")
    print(f"  {'query':<8}{'GT':<12}{'gt_cat':<18}{'rank':<6}top1")
    for qid, gt, gtc, rank, top1 in r["detail"]:
        flag = "✅" if rank == 1 else ("•" if rank != "—" else "❌")
        print(f"  {qid:<8}{gt:<12}{str(gtc):<18}{str(rank):<6}{top1} {flag}")
print("=" * 70)

ℹ️  Evaluasi LEAVE-ONE-OUT berbasis KATEGORI (konsisten dgn Tahap 5)

  METHOD=TFIDF   | Hit@5(kategori)=0.6 | MRR=0.358
  query   GT          gt_cat            rank  top1
  q01     case_026    Pasal 25          —     case_013 ❌
  q02     case_024    Pasal 114         1     case_040 ✅
  q03     case_020    Pasal 114         4     case_031 •
  q04     case_012    Pasal 111         —     case_001 ❌
  q05     case_005    Pasal 114         3     case_034 •
  q06     case_046    Pasal 114         1     case_043 ✅
  q07     case_027    Pasal 114         2     case_042 •
  q08     case_010    Pasal 112         2     case_023 •
  q09     case_030    Pasal 609         —     case_001 ❌
  q10     case_017    Pasal 609         —     case_009 ❌
  METHOD=BERT    | Hit@5(kategori)=0.6 | MRR=0.44
  query   GT          gt_cat            rank  top1
  q01     case_026    Pasal 25          —     case_032 ❌
  q02     case_024    Pasal 114         5     case_042 •
  q03     case_020    Pasal 114         1  

## Cell 28 — vi. Ekspor Script `03_retrieval.py`

In [ ]:
script_code = r'''#!/usr/bin/env python3
# ============================================================================
# 03_retrieval.py  --  Tahap 3: Case Retrieval (CBR putusan pengadilan)
# ----------------------------------------------------------------------------
# Membaca cases.csv (hasil Tahap 2), membangun representasi vektor TF-IDF dan
# (opsional) IndoBERT embedding, melatih classifier SVM & Naive Bayes pada
# TF-IDF, lalu menyediakan fungsi retrieve(query, k) berbasis cosine-similarity.
#
# Contoh pemakaian:
#   python 03_retrieval.py --cases processed/cases.csv \
#       --query "kepemilikan narkotika golongan i jenis sabu pasal 112"
#   python 03_retrieval.py --cases processed/cases.csv \
#       --queries eval/queries.json --method bert --bert -k 5
# ============================================================================
import argparse
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC

# Kolom konten yang digabung menjadi satu representasi teks per kasus
TEXT_COLS = [
    "jenis_perkara", "pasal", "pengadilan",
    "dakwaan_ringkas", "tuntutan", "barang_bukti",
    "pertimbangan_hukum", "amar_putusan", "top_terms",
]

STOPWORDS_ID = [
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "dalam", "adalah",
    "pada", "oleh", "ini", "itu", "tidak", "atau", "juga", "akan", "telah",
    "sebagai", "bahwa", "tersebut", "kepada", "para", "serta", "dapat", "ada",
    "sudah", "atas", "bagi", "agar", "lebih", "saat", "hal", "tentang", "tahun",
    "maka", "karena", "hingga", "antara", "terhadap", "setelah", "sebelum",
]


def preprocess(text):
    # Pra-pemrosesan ringan: lowercase + rapikan spasi & simbol aneh.
    t = str(text).lower()
    t = re.sub(r"[^a-z0-9\s./()-]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def build_corpus(df):
    cols = [c for c in TEXT_COLS if c in df.columns]
    def rep(row):
        return " ".join(str(row[c]) for c in cols if str(row[c]))
    return df.apply(rep, axis=1).map(preprocess).tolist()


class Retriever:
    # Mesin retrieval: simpan corpus + vektor TF-IDF (dan IndoBERT bila diminta).
    def __init__(self, df, use_bert=False, bert_model="indobenchmark/indobert-base-p1"):
        self.df = df.reset_index(drop=True)
        self.case_ids = self.df["case_id"].tolist()
        self.corpus = build_corpus(self.df)

        self.tfidf = TfidfVectorizer(
            stop_words=STOPWORDS_ID, ngram_range=(1, 2),
            min_df=1, max_df=0.95, sublinear_tf=True,
        )
        self.X_tfidf = self.tfidf.fit_transform(self.corpus)

        self.use_bert = use_bert
        if use_bert:
            self._init_bert(bert_model)
            self.X_bert = self._embed(self.corpus)

    def _init_bert(self, name):
        import torch
        from transformers import AutoModel, AutoTokenizer
        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(name)
        self.bert = AutoModel.from_pretrained(name).to(self.device).eval()

    def _embed(self, texts, batch_size=8, max_len=256):
        torch = self.torch
        vecs = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = [t if t else "kosong" for t in texts[i:i + batch_size]]
                enc = self.tok(
                    batch, padding=True, truncation=True,
                    max_length=max_len, return_tensors="pt",
                ).to(self.device)
                out = self.bert(**enc).last_hidden_state          # (B, T, H)
                mask = enc["attention_mask"].unsqueeze(-1).float()
                mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
                mean = torch.nn.functional.normalize(mean, p=2, dim=1)
                vecs.append(mean.cpu().numpy())
        return np.vstack(vecs)

    def retrieve(self, query, k=5, method="tfidf"):
        # 1) Pra-pemrosesan query
        q = preprocess(query)
        # 2) Hitung vektor query  3) cosine-similarity ke semua kasus
        if method == "tfidf":
            sims = cosine_similarity(self.tfidf.transform([q]), self.X_tfidf).ravel()
        elif method == "bert":
            if not self.use_bert:
                raise RuntimeError("Retriever dibuat tanpa BERT (set use_bert=True).")
            sims = cosine_similarity(self._embed([q]), self.X_bert).ravel()
        else:
            raise ValueError("method harus 'tfidf' atau 'bert'")
        # 4) Kembalikan top-k case_id
        top = np.argsort(-sims)[:k]
        return [
            {
                "rank": r + 1,
                "case_id": self.case_ids[i],
                "score": round(float(sims[i]), 4),
                "jenis_perkara": self.df.iloc[i].get("jenis_perkara", ""),
                "nomor_perkara": self.df.iloc[i].get("nomor_perkara", ""),
            }
            for r, i in enumerate(top)
        ]


def train_classifiers(df, X_tfidf, test_size=0.2, seed=42):
    labels = df["jenis_perkara"].replace("", "Lainnya").values
    idx = np.arange(len(df))
    strat = labels if min(Counter(labels).values()) >= 2 else None
    itr, ite = train_test_split(idx, test_size=test_size, random_state=seed, stratify=strat)
    res = {}
    for name, clf in {"SVM": LinearSVC(C=1.0), "NaiveBayes": ComplementNB(alpha=0.3)}.items():
        clf.fit(X_tfidf[itr], labels[itr])
        pred = clf.predict(X_tfidf[ite])
        res[name] = accuracy_score(labels[ite], pred)
        print(f"\n[{name}] akurasi test = {res[name]:.3f}")
        print(classification_report(labels[ite], pred, zero_division=0))
    return res


def evaluate(retriever, queries, k=5, method="tfidf"):
    # FIX: leave-one-out -> ground-truth dikeluarkan dari kandidat retrieval
    #      (konsisten dgn notebook Cell 27 & 35; cegah retrieval diri sendiri).
    hit, rr = 0, 0.0
    for q in queries:
        gt = q["ground_truth_case_id"]
        raw = retriever.retrieve(q["query"], k=k + 1, method=method)
        ids = [r["case_id"] for r in raw if r["case_id"] != gt][:k]
        if gt in ids:
            hit += 1
            rr += 1.0 / (ids.index(gt) + 1)
    n = max(len(queries), 1)
    return {"method": method, f"Hit@{k}": round(hit / n, 3),
            f"Recall@{k}": round(hit / n, 3), "MRR": round(rr / n, 3)}


def main():
    ap = argparse.ArgumentParser(description="Tahap 3 Case Retrieval (CBR)")
    ap.add_argument("--cases", required=True, help="path ke cases.csv")
    ap.add_argument("--queries", help="path ke queries.json untuk evaluasi")
    ap.add_argument("--query", help="teks query tunggal untuk diuji")
    ap.add_argument("-k", type=int, default=5, help="jumlah hasil top-k")
    ap.add_argument("--method", choices=["tfidf", "bert"], default="tfidf")
    ap.add_argument("--bert", action="store_true", help="aktifkan IndoBERT embedding")
    args = ap.parse_args()

    df = pd.read_csv(args.cases, encoding="utf-8-sig", dtype=str).fillna("")
    print(f"cases.csv dimuat: {len(df)} kasus")

    r = Retriever(df, use_bert=(args.bert or args.method == "bert"))
    train_classifiers(df, r.X_tfidf)

    if args.query:
        print(f"\nQuery: {args.query}")
        for h in r.retrieve(args.query, k=args.k, method=args.method):
            print(f"  #{h['rank']} {h['case_id']} score={h['score']} "
                  f"| {h['jenis_perkara']} | {h['nomor_perkara']}")

    if args.queries:
        qs = json.loads(Path(args.queries).read_text(encoding="utf-8"))["queries"]
        print("\n== Evaluasi pada queries.json ==")
        print(evaluate(r, qs, k=args.k, method=args.method))


if __name__ == "__main__":
    main()
'''

SCRIPT_OUT.write_text(script_code, encoding="utf-8")
print(f"✅ Script tersimpan -> {SCRIPT_OUT}")
print("   Jalankan lokal: python 03_retrieval.py --cases processed/cases.csv \\")
print("                    --queries eval/queries.json --method bert --bert -k 5")

✅ Script tersimpan -> /content/drive/MyDrive/SUBCPMK-3/03_retrieval.py
   Jalankan lokal: python 03_retrieval.py --cases processed/cases.csv \
                    --queries eval/queries.json --method bert --bert -k 5


---
## Catatan Tahap 3

| Komponen | Keterangan |
|----------|-----------|
| **Representasi TF-IDF** | `TfidfVectorizer` (unigram+bigram, stopword ID, `sublinear_tf`) |
| **Representasi BERT** | IndoBERT `indobenchmark/indobert-base-p1`, mean-pooling + L2-norm |
| **Splitting** | `train_test_split` 80:20 stratified per `jenis_perkara` (fallback acak bila kelas <2) |
| **Model TF-IDF** | `LinearSVC` (SVM) & `ComplementNB` (Naive Bayes) untuk klasifikasi/retrieval |
| **Model embedding** | IndoBERT + cosine-similarity untuk retrieval |
| **`retrieve(query, k, method)`** | preprocess → vektor query → cosine-sim ke semua kasus → top-k `case_id` |
| **`queries.json`** | 8–10 query uji + `ground_truth_case_id` (`eval/queries.json`) |
| **Metrik** | Hit@k, Recall@k, MRR untuk TF-IDF vs IndoBERT |

**Cara baca hasil retrieve()** — `score` = cosine-similarity (0–1). Makin tinggi makin mirip.
TF-IDF unggul saat *query* memakai istilah/pasal yang sama persis; IndoBERT unggul saat
*query* memakai parafrase/sinonim karena menangkap kemiripan makna.

> Jika `Hit@5` rendah, periksa: (1) kelengkapan kolom di `cases.csv` (Tahap 2 Cell 15),
> (2) apakah `ground_truth_case_id` di `queries.json` sudah benar, (3) coba `max_len`/`ngram_range` lebih besar.
